# VietnamNet headline classification: full-corpus baselines

The study compares four headline-classification pipelines using every eligible article from the full 212,705-record corpus. The source is [VietnamNet News on student Huỳnh Thành Đạt's Hugging Face repository](https://huggingface.co/datasets/dathuynh1108/vietnamnet-news).


## 1. Model and experimental settings

The models are a majority classifier, word and character TF-IDF with SVM, and frozen PhoBERT with logistic regression.


In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import importlib.util
import importlib.metadata


In [ ]:
requested_root = os.getenv("CO5177_REPO_ROOT")
if requested_root:
    os.chdir(Path(requested_root).expanduser().resolve())
elif Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
if not Path("configs/text.json").exists():
    checkout = Path("/content/CO5177-PFDAV")
    if not checkout.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "https://github.com/dathuynh1108/CO5177-PFDAV.git",
                str(checkout),
            ],
            check=True,
        )
    os.chdir(checkout)
REPO = Path.cwd().resolve()


In [ ]:
required = [
    "numpy",
    "pandas",
    "sklearn",
    "pyarrow",
    "matplotlib",
    "torch",
    "transformers",
    "pyvi",
    "accelerate",
    "seaborn",
]
if any(importlib.util.find_spec(name) is None for name in required):
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-r",
            "requirements-studies.txt",
            "accelerate>=1.2,<2",
            "seaborn",
        ],
        check=True,
    )
sys.path.insert(0, str(REPO / "src"))
print("Study checkout:", REPO)


In [ ]:
import json
import hashlib
import time
import platform

import joblib

from datetime import datetime, timezone

import numpy as np
import pandas as pd
%matplotlib inline
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import ConfusionMatrixDisplay

from studies.evidence import metrics, sha256


In [ ]:
with open(REPO / "configs/text.json", encoding="utf-8") as f:
    config = json.load(f)


In [ ]:
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
OUTPUT_ROOT = Path(
    os.getenv(
        "CO5177_TEXT_BASELINE_OUTPUT_ROOT",
        str(REPO / "artifacts" / "notebook-runs" / "text-baselines-full" / RUN_ID),
    )
)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=False)
(OUTPUT_ROOT / "config.json").write_text(
    json.dumps(config, ensure_ascii=False, indent=2) + "\n"
)
print(
    f"Run ID: {RUN_ID} | seed: {config['model_seed']} | split seed: {config['split_seed']}"
)
print(f"Run directory: {OUTPUT_ROOT}")


## 2. Source data and exploratory analysis

Empty headlines and duplicates are removed before splitting; conflicting-category duplicate groups are excluded.


In [ ]:
from studies.text import prepare, prepare_inputs, split_indices, embeddings

from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC


### Full cleaned corpus


In [ ]:
frame, data_audit = prepare(config)
frame = frame.sort_values("id").reset_index(drop=True)
frame, input_audit = prepare_inputs(frame, config)
(OUTPUT_ROOT / "input_audit.json").write_text(
    json.dumps(input_audit, ensure_ascii=False, indent=2) + "\n"
)
print(
    f"Source corpus: {data_audit['raw_rows']:,}; empty headlines: {data_audit['empty_headlines_removed']:,}; "
    f"conflicting-label rows: {data_audit['conflicting_category_rows_removed']:,}; "
    f"articles after cleaning: {data_audit['unique_usable_rows']:,}; cohort: {len(frame):,}"
)


### Category balance


In [ ]:
class_counts = frame["category"].value_counts().sort_index()
source_counts = pd.Series(data_audit["raw_class_counts"]).sort_index()
fig, axes = plt.subplots(1, 2, figsize=(16, 4))
source_counts.plot.bar(ax=axes[0], color="#263e35", rot=70)
axes[0].set(title="Category distribution in the source corpus", ylabel="Articles")
class_counts.reindex(source_counts.index).plot.bar(ax=axes[1], color="#b74c32", rot=70)
axes[1].set(title="All eligible articles used for modeling", ylabel="Headlines")
fig.tight_layout()
fig.savefig(OUTPUT_ROOT / "class_counts.png", dpi=160)
plt.show()
plt.close(fig)
data_audit["selected_class_counts"] = {str(k): int(v) for k, v in class_counts.items()}
(OUTPUT_ROOT / "data_audit.json").write_text(
    json.dumps(data_audit, ensure_ascii=False, indent=2) + "\n"
)


### Segmented headlines and labels


In [ ]:
texts = frame["headline_text"].to_numpy(dtype=object)
labels = sorted(frame["category"].unique())
label_ids = {label: index for index, label in enumerate(labels)}
target = frame["category"].map(label_ids).to_numpy()


## 3. Shared split, representations and fitting

Grouped stratification approximates 70/15/15 using the article model's partitions; vocabulary, IDF and classifiers fit training only.


In [ ]:
partitions = split_indices(frame, config)
train_idx = partitions["train"]
validation_idx = partitions["validation"]
test_idx = partitions["test"]
split_audit = {
    "method": "20-fold StratifiedGroupKFold; 14/3/3 folds",
    "seed": config["split_seed"],
    "grouping": "Final headline or Head-Tail article token identity",
    "rows": {name: len(positions) for name, positions in partitions.items()},
    "groups": {
        name: int(frame.iloc[positions]["input_group"].nunique())
        for name, positions in partitions.items()
    },
    "class_counts": {
        name: frame.iloc[positions]["category"].value_counts().sort_index().to_dict()
        for name, positions in partitions.items()
    },
}
(OUTPUT_ROOT / "split_audit.json").write_text(
    json.dumps(split_audit, ensure_ascii=False, indent=2) + "\n"
)
print("Train / validation / test:", len(train_idx), len(validation_idx), len(test_idx))


### Frozen PhoBERT representations

PhoBERT features use masked mean pooling with fixed encoder weights.


In [ ]:
cache_keys_before = {
    p.name for p in (REPO / "data/text-features").glob("*") if p.is_dir()
}
phobert_features, encoding_info = embeddings(texts.tolist(), config)
cache_reused = encoding_info["feature_cache_sha256"] in cache_keys_before
run_encoding_info = {**encoding_info, "verified_embedding_cache_reused": cache_reused}
print(
    "PhoBERT:",
    {
        k: encoding_info[k]
        for k in [
            "model",
            "revision",
            "device",
            "rows",
            "max_length",
            "pooling",
            "encoding_seconds",
        ]
    },
)
print("Verified embedding cache reused:", cache_reused)
(OUTPUT_ROOT / "phobert_features.json").write_text(
    json.dumps(run_encoding_info, ensure_ascii=False, indent=2) + "\n"
)


### Validation comparison of classifiers

Validation macro-F1 selects classifier hyperparameters.


In [ ]:
records, models, fit_history = [], {}, []
for name, kind, c_values in [
    ("Majority baseline", "majority", [None]),
    ("Word TF-IDF + Linear SVM", "word", config["classifier_c"]),
    ("Character TF-IDF + Linear SVM", "character", config["classifier_c"]),
    ("Frozen PhoBERT + Logistic regression", "phobert", config["classifier_c"]),
]:
    best_score, best_model, best_c = -1.0, None, None
    trials = []
    for c in c_values:
        if kind == "majority":
            model = DummyClassifier(strategy="most_frequent")
            train_features = np.ones((len(train_idx), 1))
            validation_features = np.ones((len(validation_idx), 1))
        elif kind == "phobert":
            model = LogisticRegression(
                C=c, max_iter=2000, random_state=config["model_seed"]
            )
            train_features, validation_features = (
                phobert_features[train_idx],
                phobert_features[validation_idx],
            )
        else:
            vectorizer = TfidfVectorizer(
                analyzer="word" if kind == "word" else "char_wb",
                ngram_range=(1, 2) if kind == "word" else (3, 5),
                min_df=2,
                max_df=0.98,
                max_features=config["max_features"],
                sublinear_tf=True,
            )
            classifier = LinearSVC(
                C=c, dual="auto", max_iter=5000, random_state=config["model_seed"]
            )
            model = Pipeline([("tfidf", vectorizer), ("classifier", classifier)])
            train_features, validation_features = (
                texts[train_idx],
                texts[validation_idx],
            )
        started = time.perf_counter()
        model.fit(train_features, target[train_idx])
        elapsed = time.perf_counter() - started
        validation_prediction = model.predict(validation_features)
        score = metrics(target[validation_idx], validation_prediction, labels)
        trial = {
            "C": c,
            "validation_accuracy": score["accuracy"],
            "validation_macro_f1": score["macro_f1"],
            "fit_seconds": elapsed,
        }
        trials.append(trial)
        fit_history.append({"model": name, **trial})
        print(
            f"{name} C={c}: validation macro-F1={score['macro_f1']:.4f}; fit={elapsed:.2f}s"
        )
        if score["macro_f1"] > best_score:
            best_score, best_model, best_c = score["macro_f1"], model, c
    models[name] = best_model
    records.append(
        {
            "name": name,
            "kind": kind,
            "selected_C": best_c,
            "validation_macro_f1": best_score,
            "validation_trials": trials,
        }
    )


In [ ]:
(OUTPUT_ROOT / "fit_history.json").write_text(
    json.dumps(fit_history, ensure_ascii=False, indent=2) + "\n"
)
(OUTPUT_ROOT / "splits.json").write_text(
    json.dumps(
        {
            "train": sorted(frame.iloc[train_idx]["id"].tolist()),
            "validation": sorted(frame.iloc[validation_idx]["id"].tolist()),
            "test": sorted(frame.iloc[test_idx]["id"].tolist()),
        },
        ensure_ascii=False,
    )
    + "\n"
)
split_ids = json.loads((OUTPUT_ROOT / "splits.json").read_text())
split_hashes = {
    name: hashlib.sha256("\n".join(ids).encode()).hexdigest()
    for name, ids in split_ids.items()
}
(OUTPUT_ROOT / "split_hashes.json").write_text(
    json.dumps(split_hashes, indent=2) + "\n"
)
provenance = {
    "source_sha256": {
        str(p): sha256(REPO / p)
        for p in [
            "src/studies/text.py",
            "configs/text.json",
            "notebooks/text_baselines.ipynb",
        ]
    },
    "python": sys.version,
    "packages": {
        p: importlib.metadata.version(p)
        for p in [
            "numpy",
            "pandas",
            "scikit-learn",
            "transformers",
            "torch",
            "pyvi",
            "pyarrow",
        ]
    },
}
(OUTPUT_ROOT / "provenance.json").write_text(json.dumps(provenance, indent=2) + "\n")


## 4. Held-out test evaluation

All four models use the same held-out test articles.


In [ ]:
test_target = target[test_idx]
predictions = pd.DataFrame(
    {"id": frame.iloc[test_idx]["id"].to_numpy(), "target": test_target}
)
for record in records:
    name, kind = record["name"], record["kind"]
    if kind == "majority":
        test_features = np.ones((len(test_idx), 1))
    elif kind == "phobert":
        test_features = phobert_features[test_idx]
    else:
        test_features = texts[test_idx]
    predicted = models[name].predict(test_features)
    measured = metrics(test_target, predicted, labels)
    record["test"] = measured
    predictions[name] = predicted
    joblib.dump(models[name], OUTPUT_ROOT / (kind + ".joblib"), compress=3)
predictions.to_csv(OUTPUT_ROOT / "predictions.csv", index=False)
summary = {
    "track": "text",
    "input_preparation": input_audit,
    "split_audit": split_audit,
    "run_id": RUN_ID,
    "seed": config["model_seed"],
    "split_seed": config["split_seed"],
    "split_sizes": {
        "train": len(train_idx),
        "validation": len(validation_idx),
        "test": len(test_idx),
    },
    "classes": labels,
    "models": records,
    "pretrained": run_encoding_info,
}
(OUTPUT_ROOT / "metrics.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2) + "\n"
)


### Model comparison


In [ ]:
rows = [
    {
        "Model": r["name"],
        "C": r["selected_C"],
        "Validation macro-F1": r["validation_macro_f1"],
        "Test accuracy": r["test"]["accuracy"],
        "Test macro-F1": r["test"]["macro_f1"],
    }
    for r in records
]
display(pd.DataFrame(rows).set_index("Model").round(4))


### Misclassified headlines


In [ ]:
chosen = max(records, key=lambda r: r["validation_macro_f1"])
errors = frame.iloc[test_idx][["id", "url", "category"]].copy()
errors["prediction"] = [labels[i] for i in predictions[chosen["name"]]]
errors.loc[errors["category"] != errors["prediction"]].head(30).to_csv(
    OUTPUT_ROOT / "errors.csv", index=False
)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
model_names = [r["name"] for r in records]
x = np.arange(len(model_names))
axes[0].bar(
    x - 0.18,
    [r["test"]["accuracy"] for r in records],
    0.36,
    label="Accuracy",
    color="#263e35",
)
axes[0].bar(
    x + 0.18,
    [r["test"]["macro_f1"] for r in records],
    0.36,
    label="Macro-F1",
    color="#b74c32",
)
axes[0].set_xticks(x, model_names, rotation=25, ha="right", fontsize=8)
axes[0].set_ylim(0, 1)
axes[0].set_title("Held-out test metrics")
axes[0].legend()
ConfusionMatrixDisplay(
    np.asarray(chosen["test"]["confusion"]), display_labels=labels
).plot(ax=axes[1], colorbar=False, include_values=False, xticks_rotation=90)
axes[1].set_title("Confusion matrix: " + chosen["name"])
fig.tight_layout()
fig.savefig(OUTPUT_ROOT / "test_metrics_confusion.png", dpi=160)
plt.show()
plt.close(fig)


### Categories with the lowest F1


In [ ]:
per_class = chosen["test"]["per_class"]
display(
    pd.DataFrame(
        [
            {
                "Category": label,
                "F1": per_class[label]["f1-score"],
                "Precision": per_class[label]["precision"],
                "Recall": per_class[label]["recall"],
            }
            for label in labels
        ]
    )
    .sort_values("F1")
    .head(8)
    .round(3)
)
print(
    "Predictions, classifiers, configuration, fitting history, metrics, errors and plots saved to",
    OUTPUT_ROOT,
)
(OUTPUT_ROOT / "run_complete.json").write_text(
    json.dumps(
        {
            "completed_at_utc": datetime.now(timezone.utc).isoformat(),
            "article_count": len(frame),
            "models": len(records),
        },
        indent=2,
    )
    + "\n"
)


## 5. Discussion and limitations

Results cover one split and seed. Pipeline differences do not isolate one component; near duplicates and temporal generalization remain unmeasured.


## 6. AI assistance

AI assistance: [AI_USE_REPORT.md](https://github.com/dathuynh1108/CO5177-PFDAV/blob/main/AI_USE_REPORT.md).
